# Notebook 07 — Multi-production Network Prototype

1. Imports
2. Production catalogue
3. Scrape multiple casts
4. Build production-performer dataframe
5. Validate data
6. Project performer network
7. Analyse first network statistics
8. Visualise prototype graph
9. Save outputs

In [7]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re


headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;"
        "q=0.9,image/avif,image/webp,*/*;q=0.8"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate",
    "Connection": "keep-alive",
}


def extract_performer_id(url):
    """
    Extract IBDB performer ID from URL.
    Example:
    /broadway-cast-staff/alfred-drake-4031 -> 4031
    """
    
    return url.split("-")[-1]


def extract_cast(url):
    """
    Extract Opening Night Cast from an IBDB production page.
    Returns a list of performers.
    """

    # Request page
    response = requests.get(
        url,
        headers=headers
    )

    response.raise_for_status()

    # Parse HTML
    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    # Locate Opening Night Cast section
    cast_section = soup.find(
        id="OpeningNightCast"
    )

    if cast_section is None:
        raise ValueError(
            "Opening Night Cast section not found"
        )

    # Find individual cast rows
    cast_rows = cast_section.find_all(
        "div",
        class_="row mobile-a-align"
    )

    cast = []

    for row in cast_rows:

        performer = row.find(
            "a",
            href=re.compile(
                "/broadway-cast-staff/"
            )
        )

        # Ignore rows without performer links
        if performer is None:
            continue

        columns = row.find_all(
            "div",
            class_="col m4 s12"
        )

        character = None

        if len(columns) > 1:
            character = columns[1].get_text(
                strip=True
            )

        cast.append(
            {
                "performer_id": extract_performer_id(
                    performer["href"]
                ),
                "performer_name": performer.text.strip(),
                "character": character
            }
        )

    return cast

In [33]:
productions = [
    {
        "production_id": "1285",
        "title": "Oklahoma!",
        "opening_date": "Mar 31, 1943",
        "url": "https://www.ibdb.com/broadway-production/oklahoma-1285"
    },

    {
        "production_id": "1694",
        "title": "Carousel",
        "opening_date": "Apr 19, 1945",
        "url": "https://www.ibdb.com/broadway-production/carousel-1694"
    },

    {
        "production_id": "1831",
        "title": "South Pacific",
        "opening_date": "Apr 7, 1949",
        "url": "https://www.ibdb.com/broadway-production/south-pacific-1831"
    },

    {
        "production_id": "1935",
        "title": "The King and I",
        "opening_date": "Mar 29, 1951",
        "url": "https://www.ibdb.com/broadway-production/the-king-and-i-1935"
    },

    {
        "production_id": "2054",
        "title": "Kiss Me, Kate",
        "opening_date": "Dec 30, 1948",
        "url": "https://www.ibdb.com/broadway-production/kiss-me-kate-2054"
    },

    {
        "production_id": "2421",
        "title": "Kismet",
        "opening_date": "Dec 03, 1953",
        "url": "https://www.ibdb.com/broadway-production/kismet-2421"
    }
]

In [34]:
# Scraping the productions, finding performers and how many are unique

all_rows = []

for production in productions:

    print(f"Scraping {production['title']}...")

    cast = extract_cast(production["url"])

    print(f"Found {len(cast)} performers")

    for performer in cast:
        all_rows.append({
            "production_id": production["production_id"],
            "production_title": production["title"],
            "opening_date": production["opening_date"],
            "performer_id": performer["performer_id"],
            "performer_name": performer["performer_name"],
            "character": performer["character"]
        })


production_performer_all = pd.DataFrame(all_rows)

print("\nFinished!")
print(f"Rows: {len(production_performer_all)}")
print(f"Unique performers: {production_performer_all['performer_id'].nunique()}")

Scraping Oklahoma!...
Found 35 performers
Scraping Carousel...
Found 49 performers
Scraping South Pacific...
Found 39 performers
Scraping The King and I...
Found 52 performers
Scraping Kiss Me, Kate...
Found 38 performers
Scraping Kismet...
Found 31 performers

Finished!
Rows: 244
Unique performers: 241


In [35]:
# Reveal performers in multiple productions

production_performer_all[
    production_performer_all.duplicated(
        "performer_id",
        keep=False
    )
].sort_values("performer_id")

,production_id,production_title,opening_date,performer_id,performer_name,character
0,1285,Oklahoma!,"Mar 31, 1943",4031,Alfred Drake,Curly
175,2054,"Kiss Me, Kate","Dec 30, 1948",4031,Alfred Drake,Fred Graham / Petruchio
213,2421,Kismet,"Dec 03, 1953",4031,Alfred Drake,"Public Poet, later called Hajj"
106,1831,South Pacific,"Apr 7, 1949",81219,Barbara Luna,Ngana
155,1935,The King and I,"Mar 29, 1951",81219,Barbara Luna,Royal Child


In [36]:
from itertools import combinations

edges = []

for production_id, group in production_performer_all.groupby("production_id"):

    performers = list(group["performer_id"])

    for p1, p2 in combinations(performers, 2):
        edges.append({
            "performer_1": min(p1, p2),
            "performer_2": max(p1, p2),
            "production_id": production_id
        })


performer_edges = pd.DataFrame(edges)

In [37]:
performer_edges_weighted = (
    performer_edges
    .groupby(
        [
            "performer_1",
            "performer_2"
        ]
    )
    .agg(
        shared_productions=("production_id", "count")
    )
    .reset_index()
)

In [38]:
performer_edges_weighted.sort_values(
    "shared_productions",
    ascending=False
).head(20)

,performer_1,performer_2,shared_productions
0,100390,101625,1
3334,80715,81228,1
3341,81219,81224,1
3340,81219,81223,1
3339,81219,81222,1
3338,81219,81221,1
3337,81219,81220,1
3336,80715,98240,1
3335,80715,81229,1
3333,80715,81227,1


In [39]:
performer_lookup = (
    production_performer_all[
        [
            "performer_id",
            "performer_name"
        ]
    ]
    .drop_duplicates()
)

performer_edges_named = (
    performer_edges_weighted
    .merge(
        performer_lookup,
        left_on="performer_1",
        right_on="performer_id"
    )
    .rename(
        columns={
            "performer_name": "performer_1_name"
        }
    )
    .drop(columns=["performer_id"])
    .merge(
        performer_lookup,
        left_on="performer_2",
        right_on="performer_id"
    )
    .rename(
        columns={
            "performer_name": "performer_2_name"
        }
    )
    .drop(columns=["performer_id"])
)

In [40]:
print("Nodes:", production_performer_all["performer_id"].nunique())
print("Edges:", len(performer_edges_weighted))

Nodes: 241
Edges: 5006


In [41]:
performer_counts = (
    production_performer_all
    .groupby(
        ["performer_id", "performer_name"]
    )
    .size()
    .sort_values(ascending=False)
)

performer_counts.head(20)

performer_id  performer_name   
4031          Alfred Drake         3
81219         Barbara Luna         2
100390        Patricia Dunn        1
93018         Tom McDuffie         1
89634         Herb Fields          1
90592         Glen Tetley          1
91024         Joseph Caruso        1
91026         Robert Pagent        1
91041         Edmund Howland       1
91367         Sandra Stahl         1
92740         Victor Duntiere      1
92769         Paul Olson           1
92916         Bonnie Evans         1
93015         Lew Foldes           1
93016         Christine Johnson    1
93017         Eric Mattson         1
93019         Ralph Tucker         1
88724         Edwin Clay           1
93020         Peter Birch          1
93826         Joseph Cunneff       1
dtype: int64

In [42]:
from collections import Counter

degree = Counter()

for _, row in performer_edges_named.iterrows():
    degree[row["performer_1_name"]] += 1
    degree[row["performer_2_name"]] += 1

degree_df = (
    pd.DataFrame(
        degree.items(),
        columns=["performer_name", "connections"]
    )
    .sort_values(
        "connections",
        ascending=False
    )
)

degree_df.head(20)

,performer_name,connections
20,Alfred Drake,101
84,Barbara Luna,89
73,Johnny Stewart,51
81,Robert Cortazal,51
80,Cristanta Cornejo,51
79,Bunny Warner,51
78,Baayork Lee,51
77,Stephanie Augustine,51
76,Lee Becker,51
75,Miriam Lawrence,51


In [44]:
import networkx as nx

G = nx.Graph()

for _, row in performer_edges_named.iterrows():
    G.add_edge(
        row["performer_1_name"],
        row["performer_2_name"]
    )

components = list(nx.connected_components(G))

len(components)



3

In [45]:
sorted(
    [len(c) for c in components],
    reverse=True
)[:10]

[102, 90, 49]

In [46]:
components_sorted = sorted(
    nx.connected_components(G),
    key=len,
    reverse=True
)

for i, component in enumerate(components_sorted):
    print("\nCOMPONENT", i+1)
    print("Size:", len(component))
    print(list(component)[:10])


COMPONENT 1
Size: 102
['June Graham', 'Paul Olson', 'Fred Davis', 'Sandra Stahl', 'Don Mayo', 'Ingrid Secretan', 'Jack Harwood', 'Herb Fields', 'Robert Penn', 'Carl Nelson']

COMPONENT 2
Size: 90
['Carolyn Maye', 'Betta St. John', 'Duane Camp', 'Dickinson Eastham', 'Henry Slate', 'Noel De Leon', 'Barbara Luna', 'James Maribo', 'Helena Schurgot', 'Ina Kurland']

COMPONENT 3
Size: 49
['Verlyn Webb', 'Marjory Svetlik', 'Kathleen Comegys', 'Margaretta De Valera', 'Peter Birch', 'Margaret Cuddy', 'Sonia Joroff', 'Polly Welch', 'Bambi Linn', 'Lee Lauterbur']
